In [3]:
import pandas as pd
import numpy as np
import kagglehub
import os

In [4]:
def collect_from_kaggle(kaggle_path: str) -> pd.DataFrame:
    path = kagglehub.dataset_download(kaggle_path)
    file_dir = os.listdir(path)
    df_path = f"{path}/{file_dir[0]}"
    df = pd.read_csv(df_path)

    return df

In [6]:
df = collect_from_kaggle("shashanknecrothapa/ames-housing-dataset")
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [ ]:
def check_outliers_overall(df: pd.DataFrame, col: str) -> dict:
    lq = np.percentile(df[col], 25)
    uq = np.percentile(df[col], 75)
    iqr = uq-lq
    lt = lq - (1.5*iqr)
    ut = uq + (1.5*iqr)

    lower_mask = df[col] < lt
    upper_mask = df[col] > ut
    overall_mask = (df[col] < lt) | (df[col] > ut)

    perc_under = lower_mask.sum()/len(df)
    perc_over = upper_mask.sum()/len(df)
    perc_all = overall_mask.sum()/len(df)

    perc_dict = {'percentage_under_threshold': perc_under,
                 'percentage_over_threshold': perc_over,
                 'percentage_anomalous': perc_all}

    return perc_dict

In [ ]:
def check_outliers_in_group(df: pd.DataFrame, target_col: str, group_col: str) -> dict:
    grouping_items = list(df[group_col].unique())
    outlier_warnings = {}
    for item in grouping_items:
        df_check = df.loc[df[group_col] == item]

        perc_outliers = check_outliers_overall(df_check, target_col)
        if perc_outliers['percentage_anomalous'] > 0.5:
            outlier_warnings[(target_col, item)] = perc_outliers['percentage_anomalous']

    return outlier_warnings

In [24]:
print(check_outliers_overall(df, 'SalePrice'))

{'percentage_under_threshold': np.float64(0.0), 'percentage_over_threshold': np.float64(0.04675767918088737), 'percentage_anomalous': np.float64(0.04675767918088737)}
